# Run 3 — Qwen3.5-9B + LoRA on Quest **videos** (8 frames per video)

**Before you start:** *Runtime → Change runtime type → A100 → Save.* Run the cells one by one (Shift + Enter).

**What the model sees.** Training: only synthetic walk-around clips of made-up charts (`--strict`: none of the real
genre / grade / country / species names, no real heatmap years). Testing: your real Quest videos, 8 frames each,
plus the prompt - never any values, names or row counts. Your Kaggle truth tables are used **only to score** the
answers afterwards.

| Step | What it does | Time (A100) |
|---|---|---|
| 1 | install Unsloth for Qwen3.5 (transformers 5.2) | ~4 min |
| 2 | upload `dxr_video_upload.zip` | 1 min |
| 2b | (optional) upload `ft_results.zip` → adds the Qwen3-VL runs to the comparison | 1 min |
| 3 | generate 2,000 + 60 synthetic clips (strict) | ~5 min |
| 3b | look at one training clip and one real clip | seconds |
| 4 | baseline: Qwen3.5-9B on your videos (+ best frames, + 20 synthetic clips) | ~15–25 min |
| 5 | train the video LoRA (250 steps) | ~40–70 min |
| 6 | test the fine-tuned model, same inputs as step 4 | ~15–25 min |
| 7 | comparison table | seconds |
| 8 | download | ~2 min |

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# 2 · Upload dxr_video_upload.zip (made on your Mac)
from google.colab import files
import zipfile
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall("/content/dxr")
%cd /content/dxr
!ls
!wc -l real_clips/real.jsonl
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content/dxr
# 2b · (optional) upload ft_results.zip from runs 1/2 - adds the Qwen3-VL best-frame results to the table
up2 = files.upload()
if up2:
    zipfile.ZipFile(next(iter(up2))).extractall("/content/dxr")
    !ls results_ft.csv

In [ ]:
%cd /content/dxr
# 3 · Synthetic training clips - strict: nothing from the real charts
!python finetune_video/make_video_dataset.py --strict --train 2000 --val 60

In [ ]:
%cd /content/dxr
# 3b · Look at one synthetic training clip (top) and one of your real test videos (bottom)
import json
from PIL import Image
from IPython.display import display
def strip(paths, base):
    ims = [Image.open(f"{base}/{p}").resize((240, 180)) for p in paths]
    s = Image.new("RGB", (240 * len(ims), 180), "white")
    for i, im in enumerate(ims): s.paste(im, (240 * i, 0))
    return s
t = json.loads(open("finetune_video/data/train.jsonl").readline()); print("training clip:", t["chart"]); display(strip(t["frames"], "finetune_video/data"))
r = json.loads(open("real_clips/real.jsonl").readline()); print("real test video:", r["video"]); display(strip(r["frames"], "real_clips"))

In [ ]:
%cd /content/dxr
# 4 · Baseline: Qwen3.5-9B before fine-tuning (whole video + best single frame + 20 synthetic clips)
!python finetune_video/eval_video.py --model unsloth/Qwen3.5-9B --name qwen35_base --best --val-n 20

In [ ]:
%cd /content/dxr
# 5 · Train the video LoRA (2,000 clips x 8 frames, 1 epoch = 250 steps)
!python finetune_video/train_video_lora.py --base unsloth/Qwen3.5-9B --epochs 1 --out lora_qwen35_video

In [ ]:
%cd /content/dxr
# 5b · keep a copy of the adapters in Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/dxr_lora && cp -r lora_qwen35_video /content/drive/MyDrive/dxr_lora/
!ls -lh /content/drive/MyDrive/dxr_lora/lora_qwen35_video | head

In [ ]:
%cd /content/dxr
# 6 · Test the fine-tuned model (same inputs as step 4)
!python finetune_video/eval_video.py --model lora_qwen35_video --name qwen35_lora --best --val-n 20

In [ ]:
%cd /content/dxr
# 7 · Comparison: Qwen3-VL runs (best frame, if 2b was done) vs Qwen3.5 base vs Qwen3.5 + video LoRA
!python finetune_video/eval_video.py --compare
from IPython.display import Markdown, display
display(Markdown(open("compare_video.md").read()))

In [ ]:
%cd /content/dxr
# 8 · Download results (and the adapters)
!zip -qr video_results.zip results_video.csv compare_video.md predictions/video_* lora_qwen35_video/training_info.json finetune_video/data/excluded_names.json
files.download("video_results.zip")
!zip -qr lora_qwen35_video.zip lora_qwen35_video
files.download("lora_qwen35_video.zip")

### If a step fails
* **Model not found** in step 4/5: replace `unsloth/Qwen3.5-9B` with `Qwen/Qwen3.5-9B` in that cell.
* **Out of memory** in step 5: add `--load-4bit`, or `--max-side 512`, or use `--base unsloth/Qwen3.5-4B` (then also in 4 and 6).
* **Disconnected** after 5b: run 1–3 again, `!cp -r /content/drive/MyDrive/dxr_lora/lora_qwen35_video /content/dxr/`, continue at 6.
* **"No such file"**: run `%cd /content/dxr` first.